# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship


## 1. My rule and its reason codes

I built a simple, transparent baseline rule to rank pages for content
review. The rule gives higher scores to pages that show observed warning
signals such as low CTR, poor average position, declining trend, low
engagement, and older content. Every page receives an Action Score, and
pages with the highest scores are reviewed first.

This is a decision-support rule, not a prediction of future recovery.
The purpose is to help a content reviewer prioritize limited review
capacity.

Reason codes:

RC1 – Declining trend detected

RC2 – Low click-through rate (CTR)

RC3 – Poor average search position

RC4 – Low engagement rate

RC5 – Old content that may need refreshing

A page may receive multiple reason codes when several warning signals
are observed.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same filters as previous notebooks
df = df[(df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)].copy()

print("Rows after filtering:", len(df))

Rows after filtering: 30000


## 2. Build the ranked queue (writes the CSV)

The baseline Action Score combines five observed signals into one simple
priority score. Pages with higher scores appear higher in the review
queue. The ranked queue is written to:

work/outputs/baseline_action_score.csv

This score is intended only for prioritization. It does not guarantee
that refreshing a page will improve performance.

In [3]:
import os

df["action_score"] = 0
df["reason_codes"] = ""

# RC1 - declining trend
mask = df["trend_direction"] == "down"
df.loc[mask, "action_score"] += 40
df.loc[mask, "reason_codes"] += "RC1;"

# RC2 - low CTR
mask = df["ctr"] < df["ctr"].median()
df.loc[mask, "action_score"] += 20
df.loc[mask, "reason_codes"] += "RC2;"

# RC3 - poor average position
mask = df["avg_position"] > df["avg_position"].median()
df.loc[mask, "action_score"] += 15
df.loc[mask, "reason_codes"] += "RC3;"

# RC4 - low engagement
mask = df["engagement_rate"] < df["engagement_rate"].median()
df.loc[mask, "action_score"] += 15
df.loc[mask, "reason_codes"] += "RC4;"

# RC5 - older content
mask = df["content_age_days"] > df["content_age_days"].median()
df.loc[mask, "action_score"] += 10
df.loc[mask, "reason_codes"] += "RC5;"

ranked = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
print(ranked[[
    "content_id",
    "action_score",
    "reason_codes"
]].head())

CSV written successfully.
             content_id  action_score      reason_codes
0  content_7b36d6ffe9bd            85  RC1;RC2;RC3;RC5;
1  content_a1fb4e703a9e            85  RC1;RC2;RC3;RC5;
2  content_79d9be0009f0            85  RC1;RC2;RC3;RC5;
3  content_67e65bf9e4e9            85  RC1;RC2;RC3;RC5;
4  content_d855229c3b74            85  RC1;RC2;RC3;RC5;


## 3. Top-20 review

I reviewed the top 20 ranked pages produced by the baseline rule.
Each page includes:

• recommended action

• reason code(s)

• confidence note

• one possible explanation for why the recommendation could be wrong

These recommendations should support a reviewer rather than replace
human judgment.

In [4]:
top20 = ranked.head(20).copy()

top20["recommended_action"] = "Review for content refresh"

top20["confidence_note"] = np.where(
    top20["action_score"] >= 70,
    "Higher confidence",
    "Moderate confidence"
)

top20["what_could_make_it_wrong"] = (
    "Traffic may change because of seasonality, external events, "
    "or factors not captured in this dataset."
)

display(
    top20[
        [
            "content_id",
            "action_score",
            "reason_codes",
            "recommended_action",
            "confidence_note",
            "what_could_make_it_wrong"
        ]
    ]
)

,content_id,action_score,reason_codes,recommended_action,confidence_note,what_could_make_it_wrong
0,content_7b36d6ffe9bd,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
1,content_a1fb4e703a9e,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
2,content_79d9be0009f0,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
3,content_67e65bf9e4e9,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
4,content_d855229c3b74,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
5,content_04e8c83b1fd8,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
6,content_56c595926939,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
7,content_a0744f34c5d8,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
8,content_cd0666cdd7ce,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."
9,content_78addf8844a1,85,RC1;RC2;RC3;RC5;,Review for content refresh,Higher confidence,"Traffic may change because of seasonality, ext..."


## 4. Weak picks + leakage check

Weak picks are pages that receive relatively high scores despite having
limited supporting evidence. These should be reviewed carefully before
taking action.

I also checked for leakage by confirming that no product decision fields
(health_score, priority_score, action_type, refresh_tier, etc.) exist in
the dataset and that no future-window columns were used to build the
baseline score.

The Action Score uses only observed information from the current
observation window and should therefore be considered a
decision-support ranking rather than a future prediction.

In [6]:
print("Lowest-confidence high-ranked pages")

weak = top20[top20["action_score"] < 60]

display(
    weak[
        [
            "content_id",
            "action_score",
            "reason_codes"
        ]
    ]
)

product_flags = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

found = [c for c in product_flags if c in df.columns]

print("\nProduct flags found:", found)

future_cols = [
    c for c in df.columns
    if "future" in c.lower()
    or "next" in c.lower()
]

print("Future-window columns:", future_cols)

Lowest-confidence high-ranked pages


,content_id,action_score,reason_codes



Product flags found: []
Future-window columns: []
